In [ ]:
# train_all_three_models.py
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_selection import SelectKBest, f_regression
import xgboost as xgb
import optuna
from optuna.samplers import TPESampler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# --------------------------
# 核心工具函数（保持原逻辑不变）
# --------------------------
def remove_outliers_iqr_enhanced(X, y_series, factor=8):
    """异常值处理"""
    q1 = y_series.quantile(0.25)
    q3 = y_series.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - factor * iqr
    upper_bound = q3 + factor * iqr
    non_outlier_indices = y_series[(y_series >= lower_bound) & (y_series <= upper_bound)].index
    return X.loc[non_outlier_indices], y_series.loc[non_outlier_indices]

def train_xgb_model(X_train, y_train, optimize=True, n_trials=200):
    """XGB模型训练（含超参优化）"""
    if optimize:
        def objective(trial):
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 100, 2000),
                'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
                'max_depth': trial.suggest_int('max_depth', 3, 15),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
                'subsample': trial.suggest_float('subsample', 0.3, 1.0),
                'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
                'reg_alpha': trial.suggest_float('reg_alpha', 0, 15),
                'reg_lambda': trial.suggest_float('reg_lambda', 0, 15),
                'gamma': trial.suggest_float('gamma', 0, 10),
                'objective': 'reg:squarederror',
                'random_state': 42,
                'tree_method': 'hist'
            }
            model = xgb.XGBRegressor(**params)
            scores = cross_val_score(model, X_train, y_train.ravel(), cv=5, scoring='neg_mean_squared_error')
            return np.mean(scores)
        
        study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
        study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
        model = xgb.XGBRegressor(**study.best_params, random_state=42, tree_method='hist')
    else:
        model = xgb.XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
    
    model.fit(X_train, y_train.ravel())
    return model

# --------------------------
# 批量训练三个模型（核心代码）
# --------------------------
if __name__ == "__main__":
    # 1. 配置参数（只需修改Excel路径）
    excel_path = "摇摆桥梁数据库.xlsx"  # 替换为你的数据集路径（务必确保正确）
    optimize = True  # 是否启用超参优化（建议开启）
    n_trials = 200  # 超参优化试验次数
    test_size = 0.2  # 测试集比例
    selected_feats = None  # 默认为使用所有特征；如需筛选，改为 ['feat1', 'feat2', ...]
    
    # 2. 定义三个目标变量的映射关系
    target_mapping = {
        '桥墩': 'Pier2_Disp',
        '支座': 'Bearing_Disp',
        '桥台': 'Abutment_Disp'
    }
    
    # 3. 创建模型保存文件夹（自动创建，无需手动建）
    save_folder = "pretrained_model"
    if not os.path.exists(save_folder):
        os.makedirs(save_folder)
        print(f"创建模型保存文件夹：{save_folder}")
    
    # 4. 加载数据集（只加载一次，循环训练三个模型）
    print("正在加载数据集...")
    X = pd.read_excel(excel_path, sheet_name='input')
    y_df = pd.read_excel(excel_path, sheet_name='output')
    
    # 筛选特征（如需自定义特征，修改这里）
    if selected_feats is None:
        selected_feats = X.columns.tolist()
    X_selected = X[selected_feats].copy()
    print(f"使用特征数：{len(selected_feats)} | 总样本数：{X_selected.shape[0]}")
    
    # 5. 循环训练三个目标变量的模型
    for target_label, target_var_name in target_mapping.items():
        print(f"\n=== 开始训练【{target_label}】模型 ===")
        
        try:
            # （1）准备目标变量
            y = y_df[target_var_name].values.reshape(-1, 1)
            y_series = pd.Series(y.flatten(), index=X_selected.index)
            print(f"目标变量：{target_var_name} | 非空样本数：{len(y_series.dropna())}")
            
            # （2）数据预处理（异常值处理+幂变换+特征选择+标准化）
            X_clean, y_clean = remove_outliers_iqr_enhanced(X_selected, y_series, factor=8)
            y_clean = y_clean.values.reshape(-1, 1)
            print(f"异常值处理后样本数：{X_clean.shape[0]}")
            
            pt = PowerTransformer(method='yeo-johnson', standardize=False)
            y_transformed = pt.fit_transform(y_clean)
            
            selector = SelectKBest(score_func=f_regression, k=min(11, X_clean.shape[1]))
            X_selected_final = selector.fit_transform(X_clean, y_clean.ravel())
            print(f"特征选择后维度：{X_selected_final.shape[1]}")
            
            scaler_X = StandardScaler()
            scaler_y = StandardScaler()
            X_scaled = scaler_X.fit_transform(X_selected_final)
            y_scaled = scaler_y.fit_transform(y_transformed)
            
            # （3）划分训练集（仅用于评估，模型用全量数据训练）
            X_train, X_test, y_train, y_test = train_test_split(
                X_scaled, y_scaled, test_size=test_size, random_state=42
            )
            
            # （4）训练模型（用全量预处理后的数据训练，保证模型性能）
            model = train_xgb_model(X_scaled, y_scaled, optimize=optimize, n_trials=n_trials)
            
            # （5）评估模型性能（输出训练效果）
            y_pred = model.predict(X_test)
            metrics = {
                'R2': r2_score(y_test, y_pred),
                'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
                'MAE': mean_absolute_error(y_test, y_pred),
                'MSE': mean_squared_error(y_test, y_pred)
            }
            print(f"模型性能：R2={metrics['R2']:.4f} | RMSE={metrics['RMSE']:.4f} | MAE={metrics['MAE']:.4f}")
            
            # （6）打包并保存模型（包含所有必要组件，后续可直接加载预测）
            model_package = {
                'model': model,
                'scaler_X': scaler_X,
                'scaler_y': scaler_y,
                'selector': selector,
                'pt': pt,
                'selected_feats': selected_feats,
                'target_label': target_label,
                'target_var_name': target_var_name,
                'train_metrics': metrics  # 记录训练性能，后续可显示
            }
            
            # 保存模型（文件名明确标注目标变量）
            save_path = os.path.join(save_folder, f"{target_label}_full_prediction_model.joblib")
            joblib.dump(model_package, save_path)
            print(f"【{target_label}】模型保存成功！路径：{save_path}")
        
        except Exception as e:
            print(f"【{target_label}】模型训练失败：{str(e)}")
            continue
    
    # 6. 训练完成提示
    print("\n=== 所有模型训练完成 ===")
    print(f"模型保存位置：{os.path.abspath(save_folder)}")
    print("生成的模型文件：")
    for file in os.listdir(save_folder):
        print(f"  - {file}")

创建模型保存文件夹：pretrained_model
正在加载数据集...


[I 2025-11-26 12:33:08,583] A new study created in memory with name: no-name-1e67a01d-b960-4aa6-9e36-ec86228809ba


使用特征数：39 | 总样本数：1000

=== 开始训练【桥墩】模型 ===
目标变量：Pier2_Disp | 非空样本数：1000
异常值处理后样本数：977
特征选择后维度：12


[I 2025-11-26 12:33:09,466] Trial 0 finished with value: -0.15300098782425375 and parameters: {'n_estimators': 812, 'learning_rate': 0.24517932047070642, 'max_depth': 12, 'colsample_bytree': 0.7190609389379257, 'subsample': 0.40921304830970556, 'min_child_weight': 4, 'reg_alpha': 0.8712541825229919, 'reg_lambda': 12.992642186624028, 'gamma': 6.011150117432088}. Best is trial 0 with value: -0.15300098782425375.
[I 2025-11-26 12:33:11,161] Trial 1 finished with value: -0.14631046548956833 and parameters: {'n_estimators': 1446, 'learning_rate': 0.005439667429522981, 'max_depth': 15, 'colsample_bytree': 0.8827098485602951, 'subsample': 0.44863737747479326, 'min_child_weight': 4, 'reg_alpha': 2.7510676478015075, 'reg_lambda': 4.563633644393066, 'gamma': 5.247564316322379}. Best is trial 1 with value: -0.14631046548956833.
[I 2025-11-26 12:33:12,122] Trial 2 finished with value: -0.15277667558314154 and parameters: {'n_estimators': 921, 'learning_rate': 0.01647477394109053, 'max_depth': 10, 

模型性能：R2=0.8907 | RMSE=0.3204 | MAE=0.2283
【桥墩】模型保存成功！路径：pretrained_model\桥墩_full_prediction_model.joblib

=== 开始训练【支座】模型 ===
目标变量：Bearing_Disp | 非空样本数：1000
异常值处理后样本数：979
特征选择后维度：12


[I 2025-11-26 12:38:08,850] Trial 0 finished with value: -0.15223404001082005 and parameters: {'n_estimators': 812, 'learning_rate': 0.24517932047070642, 'max_depth': 12, 'colsample_bytree': 0.7190609389379257, 'subsample': 0.40921304830970556, 'min_child_weight': 4, 'reg_alpha': 0.8712541825229919, 'reg_lambda': 12.992642186624028, 'gamma': 6.011150117432088}. Best is trial 0 with value: -0.15223404001082005.
[I 2025-11-26 12:38:10,356] Trial 1 finished with value: -0.15435024517943632 and parameters: {'n_estimators': 1446, 'learning_rate': 0.005439667429522981, 'max_depth': 15, 'colsample_bytree': 0.8827098485602951, 'subsample': 0.44863737747479326, 'min_child_weight': 4, 'reg_alpha': 2.7510676478015075, 'reg_lambda': 4.563633644393066, 'gamma': 5.247564316322379}. Best is trial 0 with value: -0.15223404001082005.
[I 2025-11-26 12:38:11,215] Trial 2 finished with value: -0.16530053020534222 and parameters: {'n_estimators': 921, 'learning_rate': 0.01647477394109053, 'max_depth': 10, 

模型性能：R2=0.9107 | RMSE=0.3071 | MAE=0.2113
【支座】模型保存成功！路径：pretrained_model\支座_full_prediction_model.joblib

=== 开始训练【桥台】模型 ===
目标变量：Abutment_Disp | 非空样本数：1000
异常值处理后样本数：979
特征选择后维度：12


[I 2025-11-26 12:42:50,024] Trial 0 finished with value: -0.15494670192428064 and parameters: {'n_estimators': 812, 'learning_rate': 0.24517932047070642, 'max_depth': 12, 'colsample_bytree': 0.7190609389379257, 'subsample': 0.40921304830970556, 'min_child_weight': 4, 'reg_alpha': 0.8712541825229919, 'reg_lambda': 12.992642186624028, 'gamma': 6.011150117432088}. Best is trial 0 with value: -0.15494670192428064.
[I 2025-11-26 12:42:51,407] Trial 1 finished with value: -0.1558019178380563 and parameters: {'n_estimators': 1446, 'learning_rate': 0.005439667429522981, 'max_depth': 15, 'colsample_bytree': 0.8827098485602951, 'subsample': 0.44863737747479326, 'min_child_weight': 4, 'reg_alpha': 2.7510676478015075, 'reg_lambda': 4.563633644393066, 'gamma': 5.247564316322379}. Best is trial 0 with value: -0.15494670192428064.
[I 2025-11-26 12:42:52,246] Trial 2 finished with value: -0.16551315625115898 and parameters: {'n_estimators': 921, 'learning_rate': 0.01647477394109053, 'max_depth': 10, '

模型性能：R2=0.9184 | RMSE=0.2939 | MAE=0.2050
【桥台】模型保存成功！路径：pretrained_model\桥台_full_prediction_model.joblib

=== 所有模型训练完成 ===
模型保存位置：c:\Users\Lei\Desktop\毕设\新建文件夹\三次修改\pretrained_model
生成的模型文件：
  - 支座_full_prediction_model.joblib
  - 桥台_full_prediction_model.joblib
  - 桥墩_full_prediction_model.joblib
